In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import csv

# Loading Data

In [3]:
# Load features dataset
features_df = pd.read_csv(r'C:\Users\HP\jupyter notebook files\FIFS_IPL\data\processed_data\all_league_bat_features.csv')
features_df['date'] = pd.to_datetime(features_df['date'])

# Load original batting data
batting_df = pd.read_csv(r'C:\Users\HP\jupyter notebook files\FIFS_IPL\data\mbm\all_league\all_league_bat_final.csv')
batting_df['date'] = pd.to_datetime(batting_df['date'])


In [4]:
features_df[features_df.career_avg.isnull()].shape

(8390, 35)

# Sorting the data

In [5]:
# Sort both dataframes by date
features_df.sort_values(['batter_id', 'date', 'inns'], inplace=True)
batting_df.sort_values(['batter_id', 'date', 'inns'], inplace=True)

# Convert isOut to numeric format in batting data
batting_df['isOut'] = batting_df['isOut'].replace('DNB', 'False')
batting_df['isOut'] = batting_df['isOut'].astype(str).map({'True': 1, 'False': 0}).fillna(0).astype(int)

# Add boundaries column to batting data
batting_df['boundaries'] = batting_df['fours'] + batting_df['sixes']

# Filling NULL values for league debuts

In [6]:
def fill_league_debut_nulls(features_df, batting_df):
    """Fill NULL values in features dataframe for league debuts using prior performance data.

    Args:
        features_df: DataFrame containing batting features with NULL values
        batting_df: DataFrame containing original batting performance data

    Returns:
        Updated features DataFrame with filled NULL values
    """
    # Create a copy to avoid modifying the original
    features_df = features_df.copy()

    # Get unique player IDs
    unique_players = features_df['batter_id'].unique()
    print(f"Processing {len(unique_players)} unique players")

    # Counter for tracking updates
    updates_made = 0
    league_debuts_found = 0

    # Set minimum match threshold for reliable stats
    min_match_threshold = 3

    # Process each player
    for i, player_id in enumerate(unique_players):
        if (i+1) % 100 == 0 or i+1 == len(unique_players):
            print(f"Processed {i+1}/{len(unique_players)} players")

        # Get player data from features
        player_features = features_df[features_df['batter_id'] == player_id].copy()

        # Skip if player only has one entry (absolute debut)
        if len(player_features) <= 1:
            continue

        # Get all player's batting records
        player_batting = batting_df[batting_df['batter_id'] == player_id].copy()

        # Check for league debuts (nulls in career stats)
        league_debuts = player_features[pd.isna(player_features['career_avg'])]

        # Skip if no league debuts found after absolute debut
        if len(league_debuts) <= 1:  # First one is absolute debut, which should stay NULL
            continue

        league_debuts_found += len(league_debuts) - 1  # Don't count absolute debut

        # Process each league debut (skip absolute debut)
        for idx, debut in league_debuts.iterrows():
            # Skip the absolute debut (first match ever)
            if debut['date'] == player_features['date'].min():
                continue

            # Get all matches before this date
            prior_matches = player_batting[player_batting['date'] < debut['date']]

            if len(prior_matches) > 0:
                # Calculate career stats based on prior matches
                total_runs = prior_matches['runs'].sum()
                total_dismissals = prior_matches['isOut'].sum()
                total_matches = len(prior_matches)
                total_balls = prior_matches['balls'].sum()
                total_boundaries = prior_matches['boundaries'].sum()

                # Calculate career stats
                career_avg = total_runs / max(total_dismissals, 1)
                career_rpm = total_runs / total_matches
                career_sr = (total_runs * 100) / max(total_balls, 1)
                career_bpm = total_boundaries / total_matches
                career_bfpm = total_balls / total_matches

                # Update the features dataframe
                features_df.loc[idx, 'career_avg'] = career_avg
                features_df.loc[idx, 'career_rpm'] = career_rpm
                features_df.loc[idx, 'career_sr'] = career_sr
                features_df.loc[idx, 'career_bpm'] = career_bpm
                features_df.loc[idx, 'career_bfpm'] = career_bfpm

                updates_made += 1

                # If innings is the same, we can also update innings stats
                prior_inns_matches = prior_matches[prior_matches['inns'] == debut['inns']]
                if len(prior_inns_matches) > 0:
                    inns_runs = prior_inns_matches['runs'].sum()
                    inns_dismissals = prior_inns_matches['isOut'].sum()
                    inns_matches = len(prior_inns_matches)
                    inns_balls = prior_inns_matches['balls'].sum()
                    inns_boundaries = prior_inns_matches['boundaries'].sum()

                    inns_avg = inns_runs / max(inns_dismissals, 1)
                    inns_rpm = inns_runs / inns_matches
                    inns_sr = (inns_runs * 100) / max(inns_balls, 1)
                    inns_bpm = inns_boundaries / inns_matches
                    inns_bfpm = inns_balls / inns_matches

                    features_df.loc[idx, 'inns_avg'] = inns_avg
                    features_df.loc[idx, 'inns_rpm'] = inns_rpm
                    features_df.loc[idx, 'inns_sr'] = inns_sr
                    features_df.loc[idx, 'inns_bpm'] = inns_bpm
                    features_df.loc[idx, 'inns_bfpm'] = inns_bfpm

                # VENUE STATS - UPDATED LOGIC
                # For new venue, use overall stats from all prior matches
                prior_venue_matches = prior_matches[prior_matches['venue'] == debut['venue']]
                if len(prior_venue_matches) > 0:
                    # Player has played at this venue before - use venue-specific stats
                    venue_runs = prior_venue_matches['runs'].sum()
                    venue_dismissals = prior_venue_matches['isOut'].sum()
                    venue_matches = len(prior_venue_matches)
                    venue_balls = prior_venue_matches['balls'].sum()
                    venue_boundaries = prior_venue_matches['boundaries'].sum()
                else:
                    # First time at this venue - use overall stats
                    venue_runs = total_runs
                    venue_dismissals = total_dismissals
                    venue_matches = total_matches
                    venue_balls = total_balls
                    venue_boundaries = total_boundaries

                ven_avg = venue_runs / max(venue_dismissals, 1)
                ven_rpm = venue_runs / venue_matches
                ven_sr = (venue_runs * 100) / max(venue_balls, 1)
                ven_bpm = venue_boundaries / venue_matches
                ven_bfpm = venue_balls / venue_matches

                features_df.loc[idx, 'ven_avg'] = ven_avg
                features_df.loc[idx, 'ven_rpm'] = ven_rpm
                features_df.loc[idx, 'ven_sr'] = ven_sr
                features_df.loc[idx, 'ven_bpm'] = ven_bpm
                features_df.loc[idx, 'ven_bfpm'] = ven_bfpm

                # SEASON STATS - UPDATED LOGIC
                if 'season' in features_df.columns and not pd.isna(debut['season']):
                    season = debut['season']
                    debut_league = debut['league']

                    # Variable to track if we found valid season data
                    found_season_data = False

                    # Check if player has played in this league before
                    prior_league_matches = prior_matches[prior_matches['league'] == debut_league]

                    if len(prior_league_matches) > 0:
                        # If this is not the first season in this league
                        # Use the most recent previous season in this league (and only that)
                        if isinstance(season, (int, float)):
                            # Get all seasons played in this league
                            league_seasons = sorted(prior_league_matches['date'].dt.year.unique())
                            # Find the most recent previous season
                            prev_seasons = [s for s in league_seasons if s < season]
                            if prev_seasons:
                                prev_season = max(prev_seasons)
                                # Get matches from previous season in this league
                                prev_season_matches = prior_league_matches[prior_league_matches['date'].dt.year == prev_season]

                                if len(prev_season_matches) > 0:
                                    # Use previous season's data
                                    season_runs = prev_season_matches['runs'].sum()
                                    season_dismissals = prev_season_matches['isOut'].sum()
                                    season_matches = len(prev_season_matches)
                                    season_balls = prev_season_matches['balls'].sum()
                                    season_boundaries = prev_season_matches['boundaries'].sum()
                                    found_season_data = True
                    else:
                        # First season in this league - look at most recently played league with sufficient matches

                        # Group prior matches by league and get the most recent date for each league
                        league_data = prior_matches.groupby('league').agg({
                            'date': 'max',
                            'batter_id': 'count'
                        }).rename(columns={'batter_id': 'match_count'})

                        # Sort leagues by most recent play date
                        league_data = league_data.sort_values('date', ascending=False)

                        # Try each league in order of recency until finding one with enough matches
                        for league, data in league_data.iterrows():
                            if data['match_count'] >= min_match_threshold:
                                # Found a league with enough matches
                                league_matches = prior_matches[prior_matches['league'] == league]

                                season_runs = league_matches['runs'].sum()
                                season_dismissals = league_matches['isOut'].sum()
                                season_matches = len(league_matches)
                                season_balls = league_matches['balls'].sum()
                                season_boundaries = league_matches['boundaries'].sum()
                                found_season_data = True
                                break

                        # If no league with sufficient matches was found,
                        # try using current season data from other leagues
                        if not found_season_data and isinstance(season, (int, float)):
                            same_season_matches = prior_matches[prior_matches['date'].dt.year == season]

                            if len(same_season_matches) > 0:
                                # Use matches from the same season across all leagues
                                season_runs = same_season_matches['runs'].sum()
                                season_dismissals = same_season_matches['isOut'].sum()
                                season_matches = len(same_season_matches)
                                season_balls = same_season_matches['balls'].sum()
                                season_boundaries = same_season_matches['boundaries'].sum()
                                found_season_data = True

                    # If we found valid season data, calculate and update season stats
                    if found_season_data:
                        ssn_avg = season_runs / max(season_dismissals, 1)
                        ssn_rpm = season_runs / season_matches
                        ssn_sr = (season_runs * 100) / max(season_balls, 1)
                        ssn_bpm = season_boundaries / season_matches
                        ssn_bfpm = season_balls / season_matches

                        features_df.loc[idx, 'ssn_avg'] = ssn_avg
                        features_df.loc[idx, 'ssn_rpm'] = ssn_rpm
                        features_df.loc[idx, 'ssn_sr'] = ssn_sr
                        features_df.loc[idx, 'ssn_bpm'] = ssn_bpm
                        features_df.loc[idx, 'ssn_bfpm'] = ssn_bfpm
                    # If we didn't find valid season data, the values will remain NULL
                    # and be handled by any final NULL handling step

                # Update last5 stats if we have enough prior matches
                if 'last5_avg' in features_df.columns:
                    # Use all available prior matches, up to 5
                    last5_matches = prior_matches.sort_values('date', ascending=False).head(min(5, len(prior_matches)))

                    last5_runs = last5_matches['runs'].sum()
                    last5_dismissals = last5_matches['isOut'].sum()
                    last5_matches_count = len(last5_matches)
                    last5_balls = last5_matches['balls'].sum()
                    last5_boundaries = last5_matches['boundaries'].sum()

                    last5_avg = last5_runs / max(last5_dismissals, 1)
                    last5_rpm = last5_runs / last5_matches_count
                    last5_sr = (last5_runs * 100) / max(last5_balls, 1)
                    last5_bpm = last5_boundaries / last5_matches_count
                    last5_bfpm = last5_balls / last5_matches_count

                    features_df.loc[idx, 'last5_avg'] = last5_avg
                    features_df.loc[idx, 'last5_sr'] = last5_sr
                    features_df.loc[idx, 'last5_rpm'] = last5_rpm
                    features_df.loc[idx, 'last5_bfpm'] = last5_bfpm
                    features_df.loc[idx, 'last5_bpm'] = last5_bpm

    print(f"\nFound {league_debuts_found} league debuts (after absolute debut)")
    print(f"Made {updates_made} updates to fill NULL values")

    return features_df

In [7]:
updated_features_df = fill_league_debut_nulls(features_df, batting_df)


Processing 4708 unique players
Processed 100/4708 players
Processed 200/4708 players
Processed 300/4708 players
Processed 400/4708 players
Processed 500/4708 players
Processed 600/4708 players
Processed 700/4708 players
Processed 800/4708 players
Processed 900/4708 players
Processed 1000/4708 players
Processed 1100/4708 players
Processed 1200/4708 players
Processed 1300/4708 players
Processed 1400/4708 players
Processed 1500/4708 players
Processed 1600/4708 players
Processed 1700/4708 players
Processed 1800/4708 players
Processed 1900/4708 players
Processed 2000/4708 players
Processed 2100/4708 players
Processed 2200/4708 players
Processed 2300/4708 players
Processed 2400/4708 players
Processed 2500/4708 players
Processed 2600/4708 players
Processed 2700/4708 players
Processed 2800/4708 players
Processed 2900/4708 players
Processed 3000/4708 players
Processed 3100/4708 players
Processed 3200/4708 players
Processed 3300/4708 players
Processed 3400/4708 players
Processed 3500/4708 player

# Final Processing

In [9]:
updated_features_df['date'] = pd.to_datetime(features_df['date'])
updated_features_df.sort_values(['date', 'inns'], inplace=True)

In [11]:
updated_features_df.career_avg.isnull().sum()

4708

In [12]:
def add_career_debut_column(df):
    """
    Adds a career_debut column to the dataframe.
    Sets value to 1 where career_avg is null (indicating league debut),
    and 0 for all other rows.

    Args:
        df: DataFrame containing batting or bowling features

    Returns:
        DataFrame with the additional career_debut column
    """
    # Create a copy to avoid modifying the original
    df = df.copy()

    # Create career_debut column based on whether career_avg is null
    df['career_debut'] = pd.isna(df['career_avg']).astype(int)

    print(f"Added career_debut column. Found {df['career_debut'].sum()} career debuts.")

    return df

In [13]:
# After creating your features dataframe
updated_features_df = add_career_debut_column(updated_features_df)

Added career_debut column. Found 4708 career debuts.


In [14]:
def convert_nan_to_zero(df):
    """
    Convert all NaN values in the dataframe to 0.

    Args:
        df: DataFrame containing features with potential NaN values

    Returns:
        DataFrame with NaN values replaced by 0
    """
    # Count NaN values before conversion
    nan_count_before = df.isna().sum().sum()

    # Make a copy to avoid modifying the original dataframe
    df_copy = df.copy()

    # Replace all NaN values with 0
    df_copy = df_copy.fillna(0)

    # Verify all NaNs have been replaced
    nan_count_after = df_copy.isna().sum().sum()

    print(f"Replaced {nan_count_before} NaN values with 0")
    print(f"Remaining NaN values: {nan_count_after}")

    return df_copy

# Usage:
updated_features_df = convert_nan_to_zero(updated_features_df)

Replaced 434615 NaN values with 0
Remaining NaN values: 0


In [15]:
updated_features_df

,match_id,date,inns,batter,batter_id,bat_team,bowl_team,venue,league,career_avg,...,ssn_rpm,ssn_sr,ssn_bpm,ssn_bfpm,last5_avg,last5_sr,last5_rpm,last5_bfpm,last5_bpm,career_debut
89289,211048,2005-02-17,1,MJ Clarke,4578,Australia,New Zealand,Eden Park,T20IS,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,1
3184,211048,2005-02-17,1,AC Gilchrist,5390,Australia,New Zealand,Eden Park,T20IS,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,1
68802,211048,2005-02-17,1,JR Hopes,5702,Australia,New Zealand,Eden Park,T20IS,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,1
88201,211048,2005-02-17,1,MEK Hussey,5939,Australia,New Zealand,Eden Park,T20IS,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,1
94361,211048,2005-02-17,1,MS Kasprowicz,6126,Australia,New Zealand,Eden Park,T20IS,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52122,1473458,2025-04-07,2,HH Pandya,625371,Mumbai Indians,Royal Challengers Bangalore,Wankhede Stadium,IPL,28.808989,...,13.000000,118.181818,1.333333,11.000000,33.666667,146.376812,20.2,13.8,2.6,0
64033,1473458,2025-04-07,2,JJ Bumrah,625383,Mumbai Indians,Royal Challengers Bangalore,Wankhede Stadium,IPL,9.714286,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0
156655,1473458,2025-04-07,2,WG Jacks,897549,Mumbai Indians,Royal Challengers Bangalore,Wankhede Stadium,IPL,26.200000,...,10.666667,103.225806,1.000000,10.333333,10.000000,102.040816,10.0,9.8,1.2,0
152061,1473458,2025-04-07,2,Tilak Varma,1170265,Mumbai Indians,Royal Challengers Bangalore,Wankhede Stadium,IPL,40.354839,...,23.750000,113.095238,2.500000,21.000000,39.666667,120.202020,23.8,19.8,2.8,0


In [17]:
# Define the cutoff date
cutoff_date = pd.to_datetime('2025-03-21')

# Filter the dataset to include only rows with dates on or before 21 march 2025
updated_features_df = updated_features_df[updated_features_df['date'] <= cutoff_date]

In [18]:
min(updated_features_df.date), max(updated_features_df.date)

(Timestamp('2005-02-17 00:00:00'), Timestamp('2025-03-21 00:00:00'))

In [22]:
updated_features_df = updated_features_df.reindex(columns=['date', 'match_id' , 'batter', 'batter_id', 'bat_team', 'bowl_team', 'league',
        'career_avg', 'career_rpm', 'career_sr', 'career_bpm', 'career_bfpm',
        'inns', 'inns_avg', 'inns_rpm', 'inns_sr', 'inns_bpm', 'inns_bfpm',
        'venue', 'ven_avg', 'ven_rpm', 'ven_sr', 'ven_bpm', 'ven_bfpm',
        'season', 'ssn_avg', 'ssn_rpm', 'ssn_sr', 'ssn_bpm', 'ssn_bfpm',
        'last5_avg', 'last5_sr', 'last5_rpm', 'last5_bfpm', 'last5_bpm',
        'career_debut'])

In [23]:
updated_features_df

,date,match_id,batter,batter_id,bat_team,bowl_team,league,career_avg,career_rpm,career_sr,...,ssn_rpm,ssn_sr,ssn_bpm,ssn_bfpm,last5_avg,last5_sr,last5_rpm,last5_bfpm,last5_bpm,career_debut
89289,2005-02-17,211048,MJ Clarke,4578,Australia,New Zealand,T20IS,0.000000,0.000000,0.000000,...,0.0,0.00,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,1
3184,2005-02-17,211048,AC Gilchrist,5390,Australia,New Zealand,T20IS,0.000000,0.000000,0.000000,...,0.0,0.00,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,1
68802,2005-02-17,211048,JR Hopes,5702,Australia,New Zealand,T20IS,0.000000,0.000000,0.000000,...,0.0,0.00,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,1
88201,2005-02-17,211048,MEK Hussey,5939,Australia,New Zealand,T20IS,0.000000,0.000000,0.000000,...,0.0,0.00,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,1
94361,2005-02-17,211048,MS Kasprowicz,6126,Australia,New Zealand,T20IS,0.000000,0.000000,0.000000,...,0.0,0.00,0.0,0.0,0.00,0.000000,0.0,0.0,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11788,2025-03-21,1443551,Abbas Afridi,1158538,Pakistan,New Zealand,T20IS,11.500000,5.111111,122.666667,...,0.0,0.00,0.0,0.0,13.00,136.842105,5.2,3.8,1.0,0
53923,2025-03-21,1443551,Haris Rauf,1161606,Pakistan,New Zealand,T20IS,6.769231,1.189189,120.547945,...,1.0,100.00,0.0,1.0,3.00,75.000000,0.6,0.8,0.0,0
57450,2025-03-21,1443551,Irfan Khan,1199426,Pakistan,New Zealand,T20IS,27.166667,14.818182,132.520325,...,6.0,80.00,1.0,7.5,10.25,128.125000,8.2,6.4,1.4,0
98162,2025-03-21,1443551,Mohammad Haris,1205559,Pakistan,New Zealand,T20IS,14.375000,14.375000,125.000000,...,5.5,68.75,1.0,8.0,10.80,108.000000,10.8,10.0,2.0,0


In [24]:
updated_features_df.to_csv(r"C:\Users\HP\jupyter notebook files\FIFS_IPL\data\null_val_handled_data\all_league_bat_features_dr.csv", index=False)